# OVERVIEW OF THE STUDY

## Parameters to sweep

Parameter Range

Material A refractive index n_A ∈ [1.5,3.0]

Material B refractive index n_B = 1.5

Extinction (both A & B) k ∈ [0, 0.2]

Design wavelength visible (e.g. 380–780 nm)

Number of layers 10 → 100

Collection half‑angle 10° → 90° → NA



## Quantities to analyze


### 1. Colour performance

Rosch–MacAdam colour solid

Optimal colour contour

CIELAB a*–b*

Colour strips

η_C, η_L heatmaps

### 2. Scattering

Differential scattering vs θ (dB)

Integrated efficiencies vs λ

Asymmetry g, albedo

Forward / side / backward (dB)

Pie chart at α = 70.5°

### 3. Near fields

|E| (dB), xz / yz / xy

Poynting flow + streamlines

Hotspots

### 4. Resonances

Mie l‑decomposition

l‑resolved spectra

### 5. Sensitivity

η_C, η_L vs n_layers, Δn, k

In [ ]:
# CELL 0 — IMPORTS & GLOBAL SETTINGS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bragg_onion._notebook_tools.io import load_sweep_outputs
from bragg_onion._notebook_tools.colour import add_display_normalized_colour_columns
from bragg_onion._notebook_tools.strip_plots import (
    select_best_etaC_by_wavelength,
    plot_best_etaC_colour_strip,
    select_strip_grid,
    plot_colour_strip_grid,
    plot_sweep_heatmap_local,
    prepare_strip_grid,
    summarize_strip_grid_cases,
    plot_colour_strip_grid,
)
from bragg_onion._notebook_tools.rosch_plots import (
    make_rosch_macadam_figures,
    add_highlight_cielab_points,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600

print("Notebook helper modules imported successfully.")

In [ ]:
from bragg_onion.materials import MaterialFileSpec, ConstantDispersion, load_materials
# Load materials
FILE_PS = r"/mnt/c/Users/SchwarzN/OneDrive - Université de Fribourg/Institution/P1_BraggSpericalPigments/Simulation/RefractiveIndices/PS(hPS)/x=0.0.txt"
FILE_P2VP = r"/mnt/c/Users/SchwarzN/OneDrive - Université de Fribourg/Institution/P1_BraggSpericalPigments/Simulation/RefractiveIndices/P2VP(PDP)/x=0.0.txt"
FILE_PMMA = r"/mnt/c/Users/SchwarzN/OneDrive - Université de Fribourg/Institution/P1_BraggSpericalPigments/Simulation/RefractiveIndices/PolymethylMethacrylate.txt"

# --- sanity check ---
for f in [FILE_PS, FILE_P2VP, FILE_PMMA]:
    print(f, "→", Path(f).exists())

material_specs = {
    "PS": MaterialFileSpec(
        name="PS",
        path=FILE_PS,
        wavelength_unit="nm",
        skiprows=2,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),
    "P2VP": MaterialFileSpec(
        name="P2VP",
        path=FILE_P2VP,
        wavelength_unit="nm",
        skiprows=2,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),
    "PMMA": MaterialFileSpec(
        name="PMMA",
        path=FILE_PMMA,
        wavelength_unit="um",
        skiprows=9,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),

}

print("Loading materials...")
loaded = load_materials(material_specs)

mat_A = loaded["PS"]
mat_B = loaded["P2VP"]
mat_C = loaded["PMMA"]

# Constant air medium as dispersion object for the surrounding medium.
medium = ConstantDispersion.from_nk("air", n=1.0, k=0.0)
# medium = 1.0

print("... materials loaded")

In [ ]:
# CELL 1 — PARAMETER SPACE DEFINITION
# CELL — EXTINCTION MODIFIERS

k_extra_vals = [0.0, 0.045]

extinction_modifiers_list = [
    {
        "target": "both_materials",
        "mode": "add",
        "profile": k_val,  # constant offset
        "name": f"k_add_{k_val:.3f}",
    }
    for k_val in k_extra_vals
]

print(extinction_modifiers_list)

design_wavelengths_nm = np.array([440, 515, 640])
n_layers_vals = [10, 20, 40, 80]
collection_NA_vals = [0.22, 0.8, 1.0]
core_thickness_factors = [0.5, 1.0, 2.0]

outer_layer = ["B"]

print("design_wavelengths_nm:", design_wavelengths_nm)
print("n_layers_vals:", n_layers_vals)
print("collection_NA_vals:", collection_NA_vals)
print("core_thickness_factors:", core_thickness_factors)
print("outer_layer:", outer_layer)

In [ ]:
# CELL 2 — MASTER PARAMETER GRID (PRIMARY SWEEP)

PARAMETER_GRID = {
    "peak_wavelength_m": design_wavelengths_nm * 1e-9,
    "n_layers": n_layers_vals,
    "outer_layer": outer_layer,
    "core_thickness_factor": core_thickness_factors,
}
    
PARAMETER_GRID


In [ ]:
# CELL 3 — OUTPUT SOURCE / NOTEBOOK SETTINGS

# Windows path to the copied workstation output directory.
# This can stay in Windows form even when the notebook runs in WSL:
# load_sweep_outputs(...) will convert it automatically if needed.
SOURCE_OUTPUT_DIR_WINDOWS = (
    r"C:\Users\SchwarzN\OneDrive - Université de Fribourg"
    r"\Institution\P1_BraggSpericalPigments\Simulation\MIE_Computation"
    r"\Scattnlay_BraggOnion\outputs\analyse_PS_P2VP"
)

# Default colour view for object-colour-style plotting:
DEFAULT_COLOUR_VIEW = "display"   # "raw", "display", or "both"

print("SOURCE_OUTPUT_DIR_WINDOWS:")
print(SOURCE_OUTPUT_DIR_WINDOWS)
print("DEFAULT_COLOUR_VIEW:", DEFAULT_COLOUR_VIEW)

In [ ]:
# CELL 4 — LOAD SWEEP OUTPUTS & ADD DISPLAY-NORMALIZED COLOURS

df_all, OUTPUT_DIR, USE_PARQUET = load_sweep_outputs(
    SOURCE_OUTPUT_DIR_WINDOWS,
    combine_if_needed=True,
    verbose=True,
)

# Working dataframe for plotting/analysis
df_plot = add_display_normalized_colour_columns(df_all, inplace=False)

print("\nLoaded dataframe summary:")
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("USE_PARQUET  :", USE_PARQUET)
print("df_all rows  :", len(df_all))
print("df_plot rows :", len(df_plot))

display(df_plot.head())

In [ ]:
# CELL 5 — MAIN PLOT
# PART A — ROSCH–MACADAM COLOUR SOLID
# ============================================================

ROSCH_COLOUR_VIEW = "both"   # "raw", "display", or "both"

generated_figures = make_rosch_macadam_figures(
    df_plot,
    colour_view=ROSCH_COLOUR_VIEW,
    max_overlay_points=2000,
    fixed_marker_size=4.0,
    marker_symbol_variable="n_layers",   # change freely
    marker_symbol_style="solid_and_line",  # "solid_only" or "solid_and_line"
    marker_edge_color="black",           # or None
    marker_edge_width=0.4,
    optimal_marker_size=4.8,
    optimal_marker_edge_color=None,
    optimal_marker_edge_width=0.7,
    save_html=True,
    output_dir=OUTPUT_DIR,
    show_figures=True,
)

print("Generated Rosch–MacAdam figure keys:", list(generated_figures.keys()))

In [ ]:
df_plot

In [ ]:
print(df_plot.columns.tolist())

In [ ]:
display(df_plot.head())

In [ ]:
# ============================================================
# PART B — COLOUR STRIP OPTION 1
# Best η_C case for each design wavelength
# ============================================================

fixed_filters_best_strip = {
    # "collection_na": 1.0,
    # "n_layers": 100,
    # "design_peak_wavelength_nm": 440,
    # "extinction_modifier_name": "k_add_0.045",
}

df_best_strip = select_best_etaC_by_wavelength(
    df_plot,   # use df_plot if you want hex_display
    fixed_filters=fixed_filters_best_strip,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
)

plot_best_etaC_colour_strip(
    df_best_strip,
    x="design_peak_wavelength_nm",
    colour_hex_col="hex",   # or "hex"
    annotate_cols=["eta_C", "eta_L"],
    title="Best η_C case for each design wavelength\n(fixed k = 0.05)",
)

plt.show()

# Print properties of the best strip dataframe for reference:
df_best_strip_print = df_best_strip.copy().sort_values("design_peak_wavelength_nm")

if "eta_C" in df_best_strip_print.columns:
    df_best_strip_print["eta_C_pct"] = (100 * df_best_strip_print["eta_C"]).round(0).astype(int)

if "eta_L" in df_best_strip_print.columns:
    df_best_strip_print["eta_L_pct"] = (100 * df_best_strip_print["eta_L"]).round(0).astype(int)

cols_to_show = [
    "design_peak_wavelength_nm",
    "case_index",
    "eta_C_pct",
    "eta_L_pct",
    "n_layers",
    "n_A",
    "k",
    "collection_angle_deg",
    "outer_layer",
    "hex_display",
]

cols_to_show = [c for c in cols_to_show if c in df_best_strip_print.columns]

display(df_best_strip_print[cols_to_show])

In [ ]:
# ============================================================
# PART C — COLOUR STRIP OPTION 2
# Multi-row strip grid:
# rows = full range of one sweep parameter
# columns = design wavelength
# ============================================================


# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
row_variable = "n_layers"   # e.g. "n_layers", 
colour_hex_col = "hex"                  # or "hex_display"

fixed_filters_grid = {
    "collection_na": 1.0,   # removed automatically if row_variable matches
    "n_layers": 100,
    "extinction_modifier_name": "k_add_0.045", # or "k_add_0.000"
}

# ------------------------------------------------------------
# Prepare data + auto-generated metadata
# ------------------------------------------------------------
df_grid, grid_meta = prepare_strip_grid(
    df_all,
    df_plot,
    row_variable=row_variable,
    fixed_filters=fixed_filters_grid,
    colour_hex_col=colour_hex_col,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
    select_best_within_cell=True,
)

# Optional info messages
if grid_meta["removed_filter"] is not None:
    print(
        f"Removed '{row_variable}={grid_meta['removed_filter']}' from fixed_filters "
        f"because the row variable must vary across rows."
    )

print("Fixed filters used:")
print(grid_meta["fixed_filters_clean"])
print(f"Row variable : {grid_meta['row_variable']}")
print(f"Colour column: {grid_meta['colour_hex_col']}")
print(f"Source df    : {grid_meta['df_source_name']}")
print(f"Selected tile cases: {len(df_grid)}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plot_colour_strip_grid(
    df_grid,
    row_variable=grid_meta["row_variable"],
    x="design_peak_wavelength_nm",
    colour_hex_col=grid_meta["colour_hex_col"],
    annotate_cols=["eta_C", "eta_L"],
    row_label_fmt=grid_meta["row_label_fmt"],
    title=grid_meta["title"],
    overlay_line_cols=["eta_C", "eta_L"],
    overlay_line_colors={"eta_C": "black", "eta_L": "white"},
    overlay_line_scale="global_shared",
    overlay_reference_values=[0.0, 1.0],
)

plt.show()

# ------------------------------------------------------------
# Compact summary table of the plotted tiles
# ------------------------------------------------------------
df_grid_print = summarize_strip_grid_cases(
    df_grid,
    row_variable=grid_meta["row_variable"],
    fixed_filters=grid_meta["fixed_filters_clean"],
    colour_hex_col=grid_meta["colour_hex_col"],
    x="design_peak_wavelength_nm",
)

display(df_grid_print.sort_values("design_peak_wavelength_nm"))

In [ ]:
# (1.2) CIELAB a*–b* plane
plt.figure(figsize=(6,6))
plt.scatter(df_all["a"], df_all["b"], c=df_all["hex"], s=15)
plt.axhline(0, color="gray")
plt.axvline(0, color="gray")
plt.xlabel("a*")
plt.ylabel("b*")
plt.title("CIELAB a*–b* plane")
plt.axis("equal")
plt.show()

CELL 6 — η_C AND η_L HEATMAPS (1)

In [ ]:
subset = df_all[
    # (df_all["n_layers"] == 80) &
    (df_all["extinction_modifier_name"] == "k_add_0.045") &
    (np.isclose(df_all["collection_angle_deg"], 90.0))
]

from bragg_onion.sweep import plot_sweep_heatmap

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_C",
)
plt.title("η_C heatmap")
plt.show()

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_L",
)
plt.title("η_L heatmap")
plt.show()

CELL 7 — SCATTERING ANALYSIS (2.0)

In [ ]:
from bragg_onion._notebook_tools.case_selection import (
    select_single_case_by_filters,
)

filters = {
    "extinction_modifier_name": "k_add_0.045",
    "n_layers": 80,
    "design_peak_wavelength_nm": 440.0,
    "collection_na": 1.0,
    "core_thickness_factor": 2.0,   # ✅ choose one
}

row_case = select_single_case_by_filters(df_plot, filters)
display(row_case)

In [ ]:
from bragg_onion.geometry import ExtinctionModifier

K_EXTRA_VALS = [0.0, 0.045]

EXTINCTION_MODIFIERS_LIST = [
    ExtinctionModifier(
        target="both_materials",
        mode="add",
        profile=k_val,
        name=f"k_add_{k_val:.3f}",
    )
    for k_val in K_EXTRA_VALS
]

In [ ]:
from bragg_onion.sweep import run_bragg_onion_sweep
from bragg_onion._notebook_tools.field_analysis import rebuild_stack_from_row

analysis_wavelengths_nm = np.arange(380, 781, 1)
analysis_theta_deg = np.linspace(0, 180, 721)

# ✅ reconstruct stack exactly like sweep
geometry_sel, stack_sel = rebuild_stack_from_row(
    row_case,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs={},   # same as sweep
)

modifier_name = row_case["extinction_modifier_name"]

modifier = next(
    m for m in EXTINCTION_MODIFIERS_LIST if m.name == modifier_name
)

sweep_case = run_bragg_onion_sweep(
    geometry_mode="peak_wavelength",
    parameter_grid={
        "peak_wavelength_m": [float(row_case["peak_wavelength_m"])],
        "n_layers": [int(row_case["n_layers"])],   # ✅ FIX
        "outer_layer": [str(row_case["outer_layer"])],
        "core_thickness_factor": [float(row_case["core_thickness_factor"])],
    },
    material_a=mat_A,
    material_b=mat_B,
    extinction_modifiers=[modifier],
    n_medium=1.0,
    wavelengths_m=analysis_wavelengths_nm * 1e-9,
    theta_rad=np.deg2rad(analysis_theta_deg),
    collection_na=float(row_case["collection_na"]),
    collection_direction="backward",
    integration_quantity_for_colour="q_collected_geom",
    colour_normalize_input=False,
    store_full_results=True,
    progress=False,
)


from bragg_onion._notebook_tools.scattering_extract import extract_case_arrays

data = extract_case_arrays(sweep_case)

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import plot_differential_vs_theta

plot_differential_vs_theta(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=True,
    y_mode="db"
)

In [ ]:
import bragg_onion._notebook_tools.scattering_plots
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_plots)
from bragg_onion._notebook_tools.scattering_plots import (
    plot_polar_differential_scattering
)

plot_polar_differential_scattering(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=True,
    y_mode="db",
)

plot_polar_differential_scattering(
    data["theta_deg"],
    data["wavelengths_nm"],
    data["dcs_geom"],
    plot_all=False,
    selected_wavelengths_nm=[450, 550, 650],
    show_legend=True,
    y_mode="db",
)

In [ ]:
import bragg_onion._notebook_tools.scattering_sectors
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_sectors)
from bragg_onion._notebook_tools.scattering_sectors import (
    integrate_sectors,
    plot_scattering_sectors,
)

sector = integrate_sectors(
    np.deg2rad(data["theta_deg"]),
    data["dcs_geom"],
    alpha_deg=70.5,
)

plot_scattering_sectors(
    data["wavelengths_nm"],
    sector,
    alpha_deg=70.5,
    y_mode="db",
)

In [ ]:
from bragg_onion._notebook_tools.scattering_sectors import plot_scattering_pie


plot_scattering_pie(
    data["wavelengths_nm"],
    sector,
    wavelength_nm=row_case["design_peak_wavelength_nm"],
    alpha_deg=70.5,
    include_absorption=True,
    qsca=data["qsca"],
    qabs=data["qabs"],
)

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import (
    plot_integrated_efficiencies
)

plot_integrated_efficiencies(
    data["wavelengths_nm"],
    qsca=data["qsca"],
    qext=data["qext"],
    qabs=data["qabs"],
    wavelengths_int_nm=data["wavelengths_int_nm"],
    q_collected_geom=data["q_collected_geom"],
    row_case=row_case,
)

In [ ]:
case_result = sweep_case.case_results[0]

scat  = case_result.scattering_result
integ = case_result.integrated_result   # ✅ REQUIRED
col   = case_result.colour_result

In [ ]:
from bragg_onion._notebook_tools.scattering_plots import (
    plot_integrated_with_g_albedo
)

plot_integrated_with_g_albedo(
    data,
    row_case=row_case,
    integ=integ,
    show_diagnostics=True,
)

In [ ]:
import bragg_onion._notebook_tools.scattering_sectors
from importlib import reload
reload(bragg_onion._notebook_tools.scattering_sectors)
from bragg_onion._notebook_tools.scattering_sectors import (
    plot_backward_alpha_sweep,
)

plot_backward_alpha_sweep(
    np.deg2rad(data["theta_deg"]),
    data["dcs_geom"],
    data["wavelengths_nm"],
    alpha_list=[10, 30, 50, 70.5, 90.0],
    row_case=row_case,
    normalize=True,
)


CELL 8 — NEAR‑FIELD ANALYSIS (2.1)

In [ ]:
# Choose wavelengths & plane for field inspection
# Wavelengths for near-field inspection
field_wavelengths_nm = [
    float(row_case["design_peak_wavelength_nm"]),
]

# Optional: add colour-driving peak wavelength if available
# if "q_collected_geom_peak_wavelength_nm" in row_case:
#     field_wavelengths_nm.append(
#         float(row_case["q_collected_geom_peak_wavelength_nm"])
#     )

# Choose plane and resolution
FIELD_PLANE = "xz"     # "xy", "xz", or "yz"
FIELD_NPTS = 151
EXTENT_FACTOR = 1.2   # extent = factor × outer radius
print(field_wavelengths_nm)

In [ ]:
from bragg_onion.materials import ConstantDispersion

mat_A = loaded["PS"]
mat_B = loaded["P2VP"]

n_medium= medium

In [ ]:
from bragg_onion._notebook_tools.field_analysis import rebuild_stack_from_row

FIXED_GEOMETRY_KWARGS = {}

geometry_sel, stack_sel = rebuild_stack_from_row(
    row_case,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
)

In [ ]:
# Compute field maps
from bragg_onion.fields import compute_field_map

field_maps = {}

for wl_nm in field_wavelengths_nm:
    print(f"Computing field map at {wl_nm:.1f} nm")

    field_maps[wl_nm] = compute_field_map(
        stack=stack_sel,
        wavelength_m=wl_nm * 1e-9,
        n_medium=n_medium,
        plane=FIELD_PLANE,
        npts=FIELD_NPTS,
        extent_outer_radius_factor=EXTENT_FACTOR,
    )

print("Field maps computed:", list(field_maps.keys()))

In [ ]:
# field magnitude plots (|E|, |H|, |S|)
# A) Total field magnitude |E|
import bragg_onion.fields
from importlib import reload
reload(bragg_onion.fields)
from bragg_onion.fields import plot_field_magnitude

for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="total",
        scale="db",
        clip_percentile_low=0,
        clip_percentile_high=85,
        cmap="turbo", # viridis
    )
    plt.show()

# B) Scattered‑like |E| (very informative!)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="scattered",
        scale="db",
        clip_percentile_low=0,
        clip_percentile_high=81,
        cmap="turbo", # magma
    )
    plt.show()

#  |S| magnitude (total energy flow)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="S",
        kind="total",
        scale="db",
        floor = 1e-30,
        clip_percentile_low=0,
        clip_percentile_high=100,
        cmap="turbo", # inferno
    )
    plt.show()

In [ ]:
# Poynting streamlines
# A) Delta flow
from bragg_onion.fields import plot_poynting_streamlines

for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        # background_floor=1e-14,
        # background_cmap="inferno",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        # streamline_linewidth=0.9,
        normalize_vectors=True,
        min_speed_fraction=0,
        mask_inside_sphere=False,
    )
    plt.show()

# B) Total flow (reference)
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="total",
        background_quantity="S",
        background_kind="total",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=False,
        min_speed_fraction=0.00,
        mask_inside_sphere=False,
    )
    plt.show()

# C) Scattered‑like flow
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="scattered",
        background_quantity="S",
        background_kind="scattered",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=True,
        min_speed_fraction=0.00,
        mask_inside_sphere=False,
    )
    plt.show()

CELL 9 — MIE RESONANCES (2.2)

You already implemented this in Point 6:

- l‑resolved bar plots

- l‑resolved spectra

✅ reuse those exact cells here for:

- top cases

- representative parameter cuts

In [ ]:
# CELL 10 — SENSITIVITY ANALYSIS (OPTIONAL, RECOMMENDED)
# Example: η_C vs n_layers for different n_A.
plt.figure(figsize=(7,4.5))

for n_A in n_A_vals:
    subset = df_all[
        (df_all["n_A"] == n_A) &
        (df_all["k"] == 0.05) &
        (np.isclose(df_all["collection_angle_deg"], 70.5))
    ]
    grp = subset.groupby("n_layers")["eta_C"].mean()
    plt.plot(grp.index, grp.values, marker="o", label=f"n_A={n_A:.2f}")

plt.xlabel("Number of layers")
plt.ylabel("η_C")
plt.legend()
plt.title("Sensitivity of η_C to n_layers and refractive index contrast")
plt.show()